<a href="https://colab.research.google.com/github/Love1117/Machine_learning-Projects/blob/main/Machine_Learning%20Project/05_Advanced%20Project%20(AI)/Recommendation%20System/Product_Recommendation_System/Product_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Project Summary:** Product Recommendation System Using Cosine Similarity

#**Overview**

This project implements a content-based product recommendation system using text vectorization and cosine similarity. Product categories were transformed into numerical feature vectors, enabling the model to measure similarities between products and recommend relevant items within the same category or closely related categories.
The system successfully generated accurate product recommendations by identifying products with similar characteristics and customer relevance.


#**Aim of the Project:**
To build an intelligent recommendation engine that suggests relevant products based on category similarity.
To apply vectorization and cosine similarity techniques for product matching and discovery.
To demonstrate how recommendation systems can enhance personalization and improve user shopping experiences.

In [28]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##**Load Data**

In [29]:
import pandas as pd

product_data = pd.read_csv("/content/drive/My Drive/Datasets/bigbasket_products.csv")
product_data.head()

,Unnamed: 0,product,category,sub_category,brand,sale_price,market_price,image_url,p_url,type,eancode,rating,description
0,0,Original Disinfectant Toilet Cleaner Liquid,Cleaning & Household,All Purpose Cleaners,Harpic,489.00,534.0,https://www.bigbasket.com/media/uploads/p/s/12...,https://www.bigbasket.com/pd/1207190/harpic-or...,Toilet Cleaners,Invalid Code,4.2,Harpic All in One is the one-stop shop for all...
1,1,Disinfectant Surface & Floor Cleaner Liquid - ...,Cleaning & Household,All Purpose Cleaners,Lizol,302.00,380.0,https://www.bigbasket.com/media/uploads/p/s/24...,https://www.bigbasket.com/pd/249154/lizol-disi...,Floor & Other Cleaners,8901396115113,4.2,Lizol is India's No.1 Household Cleaning Brand...
2,2,Surface Disinfectant Spray,Cleaning & Household,All Purpose Cleaners,Savlon,256.76,318.0,https://www.bigbasket.com/media/uploads/p/s/12...,https://www.bigbasket.com/pd/1212019/savlon-su...,Disinfectant Spray & Cleaners,Invalid Code,4.4,A wide variety of high traffic surfaces such a...
3,3,Harpic Disinfectant Toilet Cleaner Original200...,Cleaning & Household,All Purpose Cleaners,bb Combo,74.48,76.0,https://www.bigbasket.com/media/uploads/p/s/12...,https://www.bigbasket.com/pd/1204983/bb-combo-...,Toilet Cleaners,Invalid Code,NaN,Harpic All in One is the one stop shop for all...
4,4,Harpic Toilet Cleaner Liquid - Original 1 L + ...,Cleaning & Household,All Purpose Cleaners,bb Combo,462.00,558.0,https://www.bigbasket.com/media/uploads/p/s/12...,https://www.bigbasket.com/pd/1208977/bb-combo-...,Toilet Cleaners,Invalid Code,NaN,Harpic\n\nHarpic is Indias No.1 Toilet Cleanin...


In [30]:
# Checking to see if my data has NaN's or Duplicates
print(product_data.isna().sum())

print(product_data.duplicated().sum())

Unnamed: 0         0
product            1
category           0
sub_category       0
brand              1
sale_price         0
market_price       0
image_url          0
p_url              0
type               0
eancode          564
rating          8663
description      115
dtype: int64
0


In [31]:
# Removing all NaN and Duplicates
product_data.drop_duplicates(inplace=True)
product_data.dropna(inplace=True)
product_data.head()

,Unnamed: 0,product,category,sub_category,brand,sale_price,market_price,image_url,p_url,type,eancode,rating,description
0,0,Original Disinfectant Toilet Cleaner Liquid,Cleaning & Household,All Purpose Cleaners,Harpic,489.00,534.0,https://www.bigbasket.com/media/uploads/p/s/12...,https://www.bigbasket.com/pd/1207190/harpic-or...,Toilet Cleaners,Invalid Code,4.2,Harpic All in One is the one-stop shop for all...
1,1,Disinfectant Surface & Floor Cleaner Liquid - ...,Cleaning & Household,All Purpose Cleaners,Lizol,302.00,380.0,https://www.bigbasket.com/media/uploads/p/s/24...,https://www.bigbasket.com/pd/249154/lizol-disi...,Floor & Other Cleaners,8901396115113,4.2,Lizol is India's No.1 Household Cleaning Brand...
2,2,Surface Disinfectant Spray,Cleaning & Household,All Purpose Cleaners,Savlon,256.76,318.0,https://www.bigbasket.com/media/uploads/p/s/12...,https://www.bigbasket.com/pd/1212019/savlon-su...,Disinfectant Spray & Cleaners,Invalid Code,4.4,A wide variety of high traffic surfaces such a...
5,5,Disinfectant Bathroom Cleaner Liquid - Lemon,Cleaning & Household,All Purpose Cleaners,Harpic,299.00,362.0,https://www.bigbasket.com/media/uploads/p/s/12...,https://www.bigbasket.com/pd/1208843/harpic-di...,Toilet Cleaners,Invalid Code,4.2,Harpic Bathroom Cleaner gives you unbeatable c...
13,13,Liquid Disinfectant Cleaner for Home - Lime Fresh,Cleaning & Household,All Purpose Cleaners,Dettol,317.30,386.0,https://www.bigbasket.com/media/uploads/p/s/12...,https://www.bigbasket.com/pd/1212007/dettol-li...,Disinfectant Spray & Cleaners,Invalid Code,4.4,Dettol Disinfectant Multipurpose Liquid cleane...


In [32]:
#Dropping un-used columns
product_data.drop(columns=["Unnamed: 0", "sub_category", "p_url", "eancode", "rating"], inplace=True)
product_data.head()

,product,category,brand,sale_price,market_price,image_url,type,description
0,Original Disinfectant Toilet Cleaner Liquid,Cleaning & Household,Harpic,489.00,534.0,https://www.bigbasket.com/media/uploads/p/s/12...,Toilet Cleaners,Harpic All in One is the one-stop shop for all...
1,Disinfectant Surface & Floor Cleaner Liquid - ...,Cleaning & Household,Lizol,302.00,380.0,https://www.bigbasket.com/media/uploads/p/s/24...,Floor & Other Cleaners,Lizol is India's No.1 Household Cleaning Brand...
2,Surface Disinfectant Spray,Cleaning & Household,Savlon,256.76,318.0,https://www.bigbasket.com/media/uploads/p/s/12...,Disinfectant Spray & Cleaners,A wide variety of high traffic surfaces such a...
5,Disinfectant Bathroom Cleaner Liquid - Lemon,Cleaning & Household,Harpic,299.00,362.0,https://www.bigbasket.com/media/uploads/p/s/12...,Toilet Cleaners,Harpic Bathroom Cleaner gives you unbeatable c...
13,Liquid Disinfectant Cleaner for Home - Lime Fresh,Cleaning & Household,Dettol,317.30,386.0,https://www.bigbasket.com/media/uploads/p/s/12...,Disinfectant Spray & Cleaners,Dettol Disinfectant Multipurpose Liquid cleane...


##**Data Exploration**

In [33]:
product_data["tags"] = product_data['category'] + ", " + product_data['brand'] + " " + product_data['type'] + ", " + product_data['description']
product_data.drop(columns=["brand", "type", "description"], inplace=True)
product_data.reset_index(drop=True, inplace=True)
products = product_data

In [34]:
products.head()

,product,category,sale_price,market_price,image_url,tags
0,Original Disinfectant Toilet Cleaner Liquid,Cleaning & Household,489.00,534.0,https://www.bigbasket.com/media/uploads/p/s/12...,"Cleaning & Household, Harpic Toilet Cleaners, ..."
1,Disinfectant Surface & Floor Cleaner Liquid - ...,Cleaning & Household,302.00,380.0,https://www.bigbasket.com/media/uploads/p/s/24...,"Cleaning & Household, Lizol Floor & Other Clea..."
2,Surface Disinfectant Spray,Cleaning & Household,256.76,318.0,https://www.bigbasket.com/media/uploads/p/s/12...,"Cleaning & Household, Savlon Disinfectant Spra..."
3,Disinfectant Bathroom Cleaner Liquid - Lemon,Cleaning & Household,299.00,362.0,https://www.bigbasket.com/media/uploads/p/s/12...,"Cleaning & Household, Harpic Toilet Cleaners, ..."
4,Liquid Disinfectant Cleaner for Home - Lime Fresh,Cleaning & Household,317.30,386.0,https://www.bigbasket.com/media/uploads/p/s/12...,"Cleaning & Household, Dettol Disinfectant Spra..."


In [35]:
 products["tags"]

,tags
0,"Cleaning & Household, Harpic Toilet Cleaners, ..."
1,"Cleaning & Household, Lizol Floor & Other Clea..."
2,"Cleaning & Household, Savlon Disinfectant Spra..."
3,"Cleaning & Household, Harpic Toilet Cleaners, ..."
4,"Cleaning & Household, Dettol Disinfectant Spra..."
...,...
18495,"Beauty & Hygiene, Sirona Intimate Wash & Care,..."
18496,"Beauty & Hygiene, Niine Sanitary Napkins, Opti..."
18497,"Beauty & Hygiene, Rica Hair Removal, This uniq..."
18498,"Beauty & Hygiene, Astaberry Hair Removal, Asta..."


In [36]:
products["tags"] = products["tags"].str.replace("\n", " ")
products["tags"] = products["tags"].str.replace("\t", " ")
products["tags"] = products["tags"].apply(lambda x: x.lower())
products["tags"][4]

'cleaning & household, dettol disinfectant spray & cleaners, dettol disinfectant multipurpose liquid cleaner provides protection to you and your family against illness-causing germs and coronavirus causing covid19 virus. proven to be >99.9% effective at inactivating sars-cov-2, the virus that causes covid19, when used at 1:12 dilutions with 5 minutes contact time when tested under dirty conditions; as per standard testing protocol. this disinfectant liquid sanitizes your home and also helps maintain your personal hygiene. it can be used in bath, laundry, floor and surface cleaning, leaving everything clean and fresh. this multipurpose disinfectant cleaner for home is recommended by the indian medical association.'

##**Importing TfidfVectorizer to vectorize our tags**

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf  = TfidfVectorizer(analyzer="word", stop_words="english")
vector = tfidf.fit_transform(products["tags"])
vector.shape

(18500, 23291)

##**Importing cosine_similarity to find the nearest word in meaning**

In [38]:
from sklearn.metrics.pairwise import cosine_similarity

In [47]:
def get_recommendations(category):
  idx = products[products["category"].str.lower() == category.lower()].index[0]
  sim_scores = sorted(list(enumerate(cosine_similarity(vector[idx], vector).flatten())), reverse=True,  key= lambda item: item[1])
  for i in sim_scores[1:11]:
    print(f"Image URL: {products.iloc[i[0]]['image_url']}\nProduct Name: {products.iloc[i[0]]['product']}\nSale Price: {products.iloc[i[0]]['sale_price']}\nMarket Price: {products.iloc[i[0]]['market_price']}\n")

In [48]:
print(get_recommendations("Cleaning & Household"))

Image URL: https://www.bigbasket.com/media/uploads/p/s/1207191_2-harpic-disinfectant-toilet-cleaner-liquid-rose.jpg
Product Name: Disinfectant Toilet Cleaner Liquid - Rose
Sale Price: 479.01
Market Price: 534.0

Image URL: https://www.bigbasket.com/media/uploads/p/mm/1200367_6-harpic-disinfectant-toilet-cleaner-liquid-rose.jpg
Product Name: Disinfectant Toilet Cleaner Liquid - Rose
Sale Price: 154.0
Market Price: 178.0

Image URL: https://www.bigbasket.com/media/uploads/p/mm/1207192_2-harpic-disinfectant-toilet-cleaner-liquid-orange.jpg
Product Name: Disinfectant Toilet Cleaner Liquid - Orange
Sale Price: 474.99
Market Price: 534.0

Image URL: https://www.bigbasket.com/media/uploads/p/mm/40202555_1-harpic-power-plus-disinfectant-toilet-cleaner.jpg
Product Name: Power Plus Disinfectant Toilet Cleaner
Sale Price: 356.0
Market Price: 356.0

Image URL: https://www.bigbasket.com/media/uploads/p/mm/40068285_8-harpic-germ-stain-blaster-disinfectant-toilet-cleaner-liquid-floral.jpg
Product Nam

In [49]:
import joblib
joblib.dump(products, "/content/drive/My Drive/Models/Advanced Project/Recommendation System/Products Recommendation/products.joblib")
joblib.dump(vector, "/content/drive/My Drive/Models/Advanced Project/Recommendation System/Products Recommendation/vector.joblib")

['/content/drive/My Drive/Models/Advanced Project/Recommendation System/Products Recommendation/vector.joblib']

#**Conclusion / Deployment Summary**

##**When deployed, this recommendation system can:**

* Recommend relevant products in real time based on user interests or selected items.

* Improve product discovery, cross-selling, and customer engagement on e-commerce platforms.

* Support personalized shopping experiences by delivering data-driven recommendations.


This project demonstrates practical expertise in recommendation systems, feature engineering, and similarity-based machine learning solutions commonly used in modern e-commerce applications.